# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [39]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [ ]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv("GEMINI_API_KEY")

# if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
#     print("API key looks good so far")
# else:
#     print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

MODEL = "gemini-3-flash-preview"
openai = OpenAI(
    api_key=api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [41]:
links = fetch_website_links("https://waitbutwhy.com")
links = links[:10]

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [47]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [49]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)[:15] ## truncate for max links.
    user_prompt += "\n".join(links)
    return user_prompt

In [50]:
print(get_links_user_prompt("https://waitbutwhy.com/"))


Here is the list of links on the website https://waitbutwhy.com/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://waitbutwhy.com
https://waitbutwhy.com/homepage
#
https://waitbutwhy.com/
https://waitbutwhy.com/wait-but-who
https://waitbutwhy.com/wait-but-who
https://waitbutwhy.com/faq
https://waitbutwhy.com/contact
https://waitbutwhy.com/archive
https://waitbutwhy.com/minis
https://waitbutwhy.com/the-shed
https://waitbutwhy.com/table
http://shop.waitbutwhy.com/
http://store.waitbutwhy.com
https://store.waitbutwhy.com/collections/new-releases


In [51]:
openai = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")


def select_relevant_links(url):
    response = openai.chat.completions.create(
        model="gemma3:1b",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)},
        ],
        response_format={"type": "json_object"},
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [ ]:
select_relevant_links("https://waitbutwhy.com/")

## note: local model is not returning result in correct format.

{'links': ['https://waitbutwhy.com',
  'https://waitbutwhy.com/homepage',
  'https://waitbutwhy.com/wait-but-who',
  'https://waitbutwhy.com/faq',
  'https://waitbutwhy.com/contact']}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [55]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links["links"]:
        result += f"\n\n### Link: {link}\n"
        result += fetch_website_contents(link)
    return result

In [56]:
print(fetch_page_and_all_relevant_links("https://waitbutwhy.com/"))

## Landing Page:

Wait But Why

Home
Menu
Homepage
about
wait but who
faq
contact
archive
minis
the shed
dinner table
store
store home
new releases
posters
phone cases
cards & wrapping paper
squishy things
men’s tees
women’s tees
coffee mugs
store support
support wbw
book
#7246 (no title)
Finn’s Cave
Everything You Should Know About Sound
31 comments
Everything You Should Know About Sound
We all know what sound is. Except actually, no we don’t. Let’s fix that.
...
Read More
114
0
Latest Posts
54 comments
The sights and sounds of Bhutan
Stories from my visit to the mysterious Himalayan kingdom
...
Read More
22
0
49 comments
Tales from Toddlerhood
Eight thoughts from a strange and harrowing world
...
Read More
45
0
94 comments
All My Thoughts After 40 Hours in the Vision Pro
Welcome to the 2030s.
...
Read More
92
0
191 comments
10 Thoughts from the Fourth Trimester
1 baby, many thoughts
...
Read More
189
0
418 comments
A Short History of My Last Six Years
On June 18, 2016, I slipped and 

In [57]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a educative website
and creates a short brochure about it for prospective students, donors and contributors.
Respond in markdown without code blocks.
Include details of knoledge domains, authors if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [58]:
def get_brochure_user_prompt(name, url):
    user_prompt = f"""
You are looking at a website called: {name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the website in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000]  # Truncate if more than 5,000 characters
    return user_prompt

In [59]:
get_brochure_user_prompt("Wait But Why ?", "https://waitbutwhy.com/")

'\nYou are looking at a website called: Wait But Why ?\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the website in markdown without code blocks.\n\n\n## Landing Page:\n\nWait But Why\n\nHome\nMenu\nHomepage\nabout\nwait but who\nfaq\ncontact\narchive\nminis\nthe shed\ndinner table\nstore\nstore home\nnew releases\nposters\nphone cases\ncards & wrapping paper\nsquishy things\nmen’s tees\nwomen’s tees\ncoffee mugs\nstore support\nsupport wbw\nbook\n#7246 (no title)\nFinn’s Cave\nEverything You Should Know About Sound\n31 comments\nEverything You Should Know About Sound\nWe all know what sound is. Except actually, no we don’t. Let’s fix that.\n...\nRead More\n114\n0\nLatest Posts\n54 comments\nThe sights and sounds of Bhutan\nStories from my visit to the mysterious Himalayan kingdom\n...\nRead More\n22\n0\n49 comments\nTales from Toddlerhood\nEight thoughts from a strange and harrowing world\n...\nRead More\n45\n0\

In [60]:
openai = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")

def create_brochure(name, url):
    response = openai.chat.completions.create(
        model="gemma3:1b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(name, url)},
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [61]:
create_brochure("Wait But Why ?", "https://waitbutwhy.com/")

Okay, here's a brochure draft based on the provided information, aiming for a tone that’s engaging and subtly suggests the website’s focus on quirky, thought-provoking content:

## Welcome to Wait But Why: Where the Curious Spark

**(Image: A captivating, slightly surreal photograph – perhaps a blurred Bhutanese landscape or a quirky illustration.)**

**Are you fascinated by the unexpected?**

Wait But Why isn't just a website – it’s a journey into the unusual, the paradoxical, and the delightfully strange. We explore ideas that bend reality, challenge assumptions, and invite you to question everything you thought you knew. 

**What’s on the Menu?**

*   **Deep Dive Discussions:** From the science of sound to the history of a small hole, we tackle topics that *really* matter – and ponder them with a playful skepticism.
*   **Thought-Provoking Articles:**  We dissect complex issues with a dash of humor and a whole lot of curiosity.
*   **Strange & Wonderful Stories:** Expect unexpected insights, fascinating accounts, and moments that will leave you scratching your head (in a good way).
*   **“The Shed” Content:** Immerse yourself in our ever-growing collection of essays and perspectives.

**Who We Are**

Wait But Why is a community of individuals who believe that curiosity is the key to understanding the world. We’re a group of thinkers, writers, and enthusiasts who share a passion for exploring unconventional ideas. 

**Ready to Question Everything?**

**Explore the Archive:** 

[Link to Archive Page]

**Don't Forget:**

*   **#7246 – An Initial Thought**
*   **Mailbag #2:**  Our latest conversation sparks many questions.

**Want to Dive Deeper?**

[Link to Homepage]

**Visit us at [Wait But Why Website]**

---

**Notes & Considerations:**

*   **Tone:** I’ve aimed for a slightly whimsical and slightly humorous tone.  It avoids being overly serious.
*   **Visuals:**  The suggested image is crucial. It needs to match the overall aesthetic of the website.
*   **Call to Action:** The link to the archive and homepage are essential.
*   **Imagery:** The brochure could easily benefit from more visual elements—illustrations or photos that evoke the website's content.

Would you like me to refine this brochure further, perhaps focusing on a specific aspect of the website’s content?

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [63]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gemma3:1b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)},
        ],
        stream=True,
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)

In [65]:
stream_brochure("Wait But Why?", "https://waitbutwhy.com/")

Okay, here’s a brochure draft based on the provided information, aiming for a tone that’s both intriguing and accessible for prospective students, donors, and contributors to Wait But Why.

---

**Welcome to Wait But Why: Explore the Sound of Reality**

**(Image: A subtly layered image – perhaps a close-up of a sound recording with intriguing visuals)**

**Discover the Unexpected.**

Wait But Why is a podcast and online community exploring the world of sound – from the subtle to the extraordinary.  We delve into the science, history, and philosophy of audio, offering unique perspectives and thought-provoking conversations.

**What’s on Offer?**

* **Listen to Our Stories:** From the mysteries of Bhutan to the transformative potential of brainwave technology, we’ve got a captivating journey of discovery.
* **Dive into Sound:** Explore everything from recording techniques to the history of audio.
* **Join the Community:** Connect with passionate audiophiles, thinkers, and innovators.
* **Get Inspired:** Gain a deeper understanding of how sound shapes our world.

**For Students:**

* **Potential for Research:**  The archive and discussions often present opportunities for exploring sound design and its applications.
* **Community & Networking:** A place to connect with like-minded individuals.
* **Thought-Provoking Content:**  Experiences and perspectives challenging conventional thinking.


**For Donors & Supporters:**

* **Investing in Curiosity:** Your support fuels our exploration of the sonic landscape.
* **Behind-the-Scenes Insights:**  Learn about our team and our mission to deepen understanding.
* **Unique Educational Resources:** Access to exclusive content and community forums.


**For Contributors:**

* **Share Your Perspective:**  Contribute original content, thought-provoking articles, or insightful audio segments.
* **Engage with Our Community:**  Participate in discussions and share your expertise.

**(Small Logo of Wait But Why)**

**Ready to Listen?**

* **Start Listening:**  [Link to Wait But Why Website]
* **Join the Conversation:** [Link to Wait But Why Forum/Community]
* **Donate:** [Link to Donation Page]

---

**Notes:**

*   I’ve emphasized the unique value proposition of the website - it’s not just about sound, it’s about *understanding* it.
*   The language is designed to be engaging and slightly mysterious, appealing to those interested in the "why" behind the content.
*   I've included clear calls to action.

Would you like me to refine this brochure further, perhaps focusing on a specific aspect (like the podcast content)?

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>